In [2]:
import os
os.chdir(os.path.join(os.getcwd(), '..'))

In [4]:
os.getcwd()

'd:\\learning\\learning\\programing\\portfolio_projects\\1_image'

In [25]:
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, Callback, ModelCheckpoint

import time 

from src.components.lightning_modules import MRIDataModule, MRIModule
from src.logging import logger



In [21]:
optuna_name = 'name'
dirpath = os.path.join('checkpoints', f'{optuna_name}')
checkpoint_callback = ModelCheckpoint(
    dirpath = dirpath,
    filename ='unet-{epoch:02d}-{val_f1_score:.2f}',
    monitor = 'val_f1_score', 
    mode = 'max',    
    verbose = True,           
    save_last = True,
    every_n_epochs = 1
)


early_stop_callback = EarlyStopping(
    monitor='val_loss',
    min_delta=0.00,
    patience=7,
    mode='min',
    verbose=True
)


In [ ]:
from pathlib import Path

In [ ]:
class ModelTrainer:


    def __init__(self,
                 learning_rate = 0.001,
                 weight_decay = 0.01,
                 inp_channels = 3,
                 first_conv_out_channels = 64,
                 num_classes = 3,
                 depth = 3,
                 n_encoder_conv_layers = 2,
                 n_decoder_conv_layers = 2,
                 kernel_sizes: list | int = 3,

                 train_csv = None,
                 dev_csv = None,
                 test_csv = None,
                 batch_size = 32,
                 num_workers = 2,
                 train_empty_mri_ratio = 0.2,
                 
                 random_state = 42, 
                 model_name = 'unet',
                 checkpoint_dir = '/models/checkpoints'):

        self.model = MRIModule(learning_rate, 
                               weight_decay, 
                               inp_channels, 
                               first_conv_out_channels, 
                               num_classes, 
                               depth, 
                               n_encoder_conv_layers, 
                               n_decoder_conv_layers, 
                               kernel_sizes)

        self.data_module = MRIDataModule(train_csv, 
                                         dev_csv, 
                                         test_csv,
                                         batch_size, 
                                         num_workers, 
                                         train_empty_mri_ratio, 
                                         random_state)

        self.model_name = model_name
        self.checkpoint_dir = checkpoint_dir
        self.callbacks = None
        self._get_def_callbacks()


    def _get_def_callbacks(self):
        dirpath = Path(self.checkpoint_dir, self.model_name)
        checkpoint_callback = ModelCheckpoint(
            dirpath = dirpath, #! dvc data base url
            filename ='unet-{epoch:02d}-{val_f1_score:.2f}',
            monitor = 'val_f1_score', 
            mode = 'max',    
            verbose = True,           
            save_last = True,
            every_n_epochs = 1
        )


        early_stop_callback = EarlyStopping(
            monitor='val_loss',
            min_delta=0.00,
            patience=7,
            mode='min',
            verbose=True
        )

        self.callbacks = [checkpoint_callback, early_stop_callback]


    def __call__(
            self,
            num_epochs = 50,
            accelerator = 'auto',
            devices = 1
        ):

        self.trainer = pl.Trainer(accelerator = accelerator, 
                   devices = devices,
                   logger = logger,
                   callbacks = self.callbacks,  
                   max_epochs = num_epochs,
                   enable_progress_bar = True,
                   enable_model_summary = True)

        self.trainer.fit(self.model, self.data_module)


    def get_best_model_info(self):

        f1_score =  self.trainer.checkpoint_callback.best_model_score
        path = checkpoint_callback.best_model_path
        
        return f1_score, path 

    

In [28]:

from tqdm.auto import tqdm
import optuna


In [ ]:
class OptunaModelSelection:
    def __init__(self, config):

        self.study = optuna.create_study(config.database_url, direction='maximize')
        self.study.best_trial
        self.config = config
        self.counter = 0

    def _objective(self, trial: optuna.Trial):

        first_conv_out_channels = trial.suggest_categorical('first_conv_out_channels', [32,64,128])
        depth = trial.suggest_int('depth', 3, 5)
        n_encoder_conv_layers = trial.suggest_int('n_encoder_conv_layers',1, 3)
        n_decoder_conv_layers = trial.suggest_int('n_decoder_conv_layers',1, 3)
        kernel_sizes = [trial.suggest_categorical(f'kernel_sizes_{i}', [3, 5]) for i in range(depth*2)]
        empty_mri_ratio = trial.suggest_float('empty_mri_ratio', 0.1, 0.3)

        model_trainer = ModelTrainer(
            learning_rate = self.config.lr,
            weight_decay = self.config.weight_decay,                     
            inp_channels = self.config.inp_channels,
            first_conv_out_channels = first_conv_out_channels,
            num_classes = self.config.num_classes,
            depth = depth,
            n_encoder_conv_layers = n_encoder_conv_layers,
            n_decoder_conv_layers = n_decoder_conv_layers,
            kernel_sizes = kernel_sizes,

            train_csv = self.config.train_csv,
            dev_csv= self.config.dev_csv,
            test_csv = self.config.test_csv,
            batch_size = self.config.batch_size, # defined by test in config entity
            num_workers = self.config.num_workers, # defined by test in config entity
            train_empty_mri_ratio = empty_mri_ratio, 
            random_state=self.config.random_state,
            model_name = f'{self.config.mdoel_name}-{self.counter}',
            checkpoint_dir= self.config.checkpoint_dir
        )

        
        model_trainer(
            num_epochs = self.config.num_epochs,
            accelerator= self.config.accelerator, # defined by test in config entity
            devices = self.config.devices # defined by test in config entity
        )


        #? Should i use this line
            #*        |
            #*        |
            #*        V
        #!  self.trainers.append(model_traiener)


        f1, _ = model_trainer.get_best_model_info()
        self.counter += 1

        return f1

    
    def __call__(self):

        self.study.optimize(lambda trial: self._objective(trial), n_trials=self.config.n_trials, show_progress_bar = True)
        self.counter = 0
